# Decision Tree — Classification

---

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i, y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

and

$$
y_i \in \{1,2,\dots,K\}
$$

The goal is to learn a model that partitions the feature space to classify inputs.

---

## 2. Model Structure

A decision tree consists of decision rules of the form

$$
x_j \le t
$$

where $j$ is the feature index and $t$ is the threshold.

Each internal node splits the data and each leaf node assigns a class label.

---

## 3. Data Splitting

The dataset is split into two parts:

$$
\mathcal{D}_{\text{left}} = \{x_i : x_{i,j} \le t\}
$$

$$
\mathcal{D}_{\text{right}} = \{x_i : x_{i,j} > t\}
$$

---

## 4. Impurity Measures

### Gini Impurity

$$
G = 1 - \sum_{k=1}^{K} p_k^2
$$

### Entropy

$$
H = - \sum_{k=1}^{K} p_k \log(p_k)
$$

where

$$
p_k = \frac{1}{N} \sum_{i=1}^{N} \mathbf{1}(y_i = k)
$$

---

## 5. Split Objective

The quality of a split is:

$$
\text{Score} = \frac{n_L}{n} I_L + \frac{n_R}{n} I_R
$$

where

$$
n_L = |\mathcal{D}_{\text{left}}|, \quad n_R = |\mathcal{D}_{\text{right}}|
$$

and $I_L, I_R$ are impurity values.

---

## 6. Feature Subsampling

A subset of features is selected:

$$
\mathcal{F} \subset \{1,2,\dots,D\}
$$

with size

$$
|\mathcal{F}| = \max\left(1, \lfloor D \cdot \text{feature\_fraction} \rfloor \right)
$$

---

## 7. Threshold Selection

Candidate thresholds are computed as:

$$
t_k = \frac{x_{(k)} + x_{(k+1)}}{2}
$$

---

## 8. Leaf Node Prediction

At a leaf node:

$$
\hat{y} = \text{mode}(y)
$$

---

## 9. Stopping Criteria

Tree growth stops if:

$$
\text{depth} \ge \text{max\_depth}
$$

or

$$
N < \text{min\_samples\_split}
$$

or

$$
|\{y_i\}| = 1
$$

---

## 10. Recursive Optimization

At each node:

$$
(j^*, t^*) = \arg\min_{j,t} \text{Score}(j,t)
$$

Then split the data and repeat recursively.

---

## 11. Prediction Rule

For a sample $x$:

$$
\text{if } x_j \le t \rightarrow \text{left subtree}
$$

$$
\text{else } \rightarrow \text{right subtree}
$$

Final prediction:

$$
\hat{y} = \text{leaf value}
$$

---



## 12. Algorithm Summary

Initialize dataset  

For each node:

$$
\text{Select features } \mathcal{F}
$$

$$
\text{Compute thresholds}
$$

$$
(j^*, t^*) = \arg\min \text{Score}
$$

$$
\text{Split data}
$$

Repeat recursively  

Stop when conditions are met and assign:

$$
\hat{y} = \text{mode}(y)
$$

---

## 13. Final Optimization Perspective

The model greedily minimizes impurity:

$$
\min_{\text{tree}} \sum_{\text{nodes}} \text{impurity}
$$

In [1]:
class LeafNode:
    """
    Leaf node of the decision tree.

    Attributes
    ----------
    value : int, float
        The predicted class label (for classification) or value (for regression)
        at the leaf.
    """
    def __init__(self, value):
        self.value = value


In [2]:
class DecisionNode:
    """
    Internal decision node of the tree.

    Attributes
    ----------
    best_feature : int
        Index of the feature used for the split.
    best_threshold : float
        Threshold value for splitting.
    left_child : LeafNode or DecisionNode
        Left subtree (samples <= threshold).
    right_child : LeafNode or DecisionNode
        Right subtree (samples > threshold).
    """
    def __init__(self, best_feature, best_threshold, left_child, right_child):
        self.best_feature = best_feature
        self.best_threshold = best_threshold
        self.left_child = left_child
        self.right_child = right_child



In [3]:
class DecisionTree:
    """
    Decision Tree Classifier using recursive binary splits.

    Parameters
    ----------
    max_depth : int
        Maximum depth of the tree.
    min_samples_split : int
        Minimum number of samples required to split a node.
    scoring : str
        Impurity measure: 'gini' for Gini index, 'entropy' for Information Gain.
    feature_fraction : float
        Fraction of features to consider at each split (for random feature selection).

    Attributes
    ----------
    root : DecisionNode or LeafNode
        Root node of the trained tree.
    """
    def __init__(self, max_depth=10, min_samples_split=1, scoring='gini', feature_fraction=1.0):

        
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.scoring = scoring
        self.feature_fraction = feature_fraction
        self.root = None

        # Validate
        if type(self.max_depth) != int or self.max_depth <= 0:
            raise ValueError('Max depth must be a positive integer')
        if type(self.min_samples_split) != int or self.min_samples_split <= 0:
            raise ValueError('Min samples split must be a positive integer')
        if self.scoring not in ['gini', 'entropy']:
            raise ValueError("Scoring must be either 'gini' or 'entropy'")

    def _stopping_condition(self, data, depth):
        """
        Check whether to stop splitting the current node.

        Stopping criteria:
        - Maximum depth reached
        - Not enough samples to split
        - Node is pure (all labels identical)
        """
        
        if depth >= self.max_depth:
            return True
        if len(data) < self.min_samples_split:
            return True
        if len(np.unique(data[:, -1])) == 1:
            return True
        return False

    def _score(self, left, right):
        """
        Compute the impurity score of a potential split.

        Parameters
        ----------
        left : np.array
            Labels in the left split.
        right : np.array
            Labels in the right split.

        Returns
        -------
        float
            Weighted impurity score of the split (lower is better).
        """
        n_left = len(left)
        n_right = len(right)
        

        # Avoid invalid splits
        if n_left == 0 or n_right == 0:
            return np.inf
            
        total = n_left + n_right

        counts_left = np.unique(left, return_counts=True)[1]
        counts_right = np.unique(right, return_counts=True)[1]

        probability_left = counts_left / np.sum(counts_left)
        probability_right = counts_right / np.sum(counts_right)

        if self.scoring == 'gini':
            score_left = 1 - np.sum(probability_left**2)
            score_right = 1 - np.sum(probability_right**2)
            
        # entropy    
        else:  
            score_left = -np.sum(probability_left * np.log(probability_left))
            
            score_right = -np.sum(probability_right * np.log(probability_right))
        # Weighted average of left and right impurity
        return (n_left * score_left + n_right * score_right) / total


 

    def _random_feature(self, data):
        """
        Randomly select a subset of features based on feature_fraction.

        Returns
        -------
        selected_features : np.array
            Indices of selected features for splitting.
        """
        
        n_features = data.shape[1] - 1  # exclude target
        
        n_selected = max(1, int(round(n_features * self.feature_fraction)))
        
        selected_features = np.random.choice(n_features, n_selected, replace=False)
        
        return selected_features


    def _find_all_threshold(self, data, selected_features):
        """
        Compute all candidate thresholds for each selected feature.

        Returns
        -------
        list of np.array
            Candidate threshold values for each feature.
        """
        
        all_thresholds = []
        for feature in selected_features:
            unique_vals = np.unique(data[:, feature])
            if len(unique_vals) <= 1:
                all_thresholds.append(np.array([]))
            else:
                # Midpoints between successive values
                successive_average = (unique_vals[1:] + unique_vals[:-1]) / 2
                all_thresholds.append(successive_average)
                
        return all_thresholds

    def _split(self, data, feature, threshold):
        """
        Split data into left and right subsets based on threshold.

        Parameters
        ----------
        data : np.array
        feature : int
        threshold : float

        Returns
        -------
        left : np.array
        right : np.array
        """
        condition = data[:, feature] <= threshold
        
        return data[condition], data[~condition]

    def _best_feature_threshold(self, data, selected_features, all_thresholds):
        """
        Find the best feature and threshold combination minimizing impurity.

        Returns
        -------
        best_feature : int
        best_threshold : float
        """
        
        best_score = np.inf
        best_feature = None
        best_threshold = None

        for i, feature in enumerate(selected_features):
            thresholds = all_thresholds[i]
            if len(thresholds) == 0:
                continue
            for threshold in thresholds:
                left, right = self._split(data, feature, threshold)
                score = self._score(left[:, -1], right[:, -1])
                if score < best_score:
                    best_score = score
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def _find_best_split(self, data):
        """
        Function to select features, compute thresholds and find best split.
        """
        selected_features = self._random_feature(data)
        
        all_thresholds = self._find_all_threshold(data, selected_features)
        
        best_feature, best_threshold = self._best_feature_threshold(data, selected_features, all_thresholds)
        
        return best_feature, best_threshold

    def _build_tree(self, data, depth):
        """
        Recursively build the decision tree.

        Returns
        -------
        LeafNode or DecisionNode
        """
        
        if self._stopping_condition(data, depth):
            # Leaf prediction: majority class
            prediction = stats.mode(data[:, -1])[0]
            return LeafNode(prediction)

        best_feature, best_threshold = self._find_best_split(data)
        left_data, right_data = self._split(data, best_feature, best_threshold)

        # If no valid split found, create leaf
        if best_feature is None:
            prediction = stats.mode(data[:, -1])[0]
            return LeafNode(value=prediction)

        left_child = self._build_tree(left_data, depth + 1)
        right_child = self._build_tree(right_data, depth + 1)

        return DecisionNode(best_feature, best_threshold, left_child, right_child)

    def fit(self, X, y):
        """
        Fit the decision tree to the training data.

        Parameters
        ----------
        X : np.array
            Feature matrix (N x D)
        y : np.array
            Labels vector (N,)
        """
        
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        data = np.hstack((X, y))
        self.root = self._build_tree(data, 0)

    def _predict_single(self, x, node):
        """
        Predict label for a single sample recursively.

        Parameters
        ----------
        x : np.array
            Feature vector
        node : DecisionNode or LeafNode

        Returns
        -------
        predicted label
        """
        if isinstance(node, LeafNode):
            return node.value
        if x[node.best_feature] <= node.best_threshold:
            return self._predict_single(x, node.left_child)
        else:
            return self._predict_single(x, node.right_child)

    def predict(self, X):
        """
        Predict labels for multiple samples.

        Parameters
        ----------
        X : np.array
            Feature matrix

        Returns
        -------
        np.array
            Predicted labels
        """
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return np.array([self._predict_single(x, self.root) for x in X])

## 1. Problem Setup

Compare the **training time** of a Decision Tree classifier when using two different **splitting criteria**:

1. **Gini Impurity**
2. **Entropy**

Generate a synthetic dataset

$$
\{(x_i, y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

is the feature vector and

$$
y_i \in \{0, 1, \dots, C-1\}
$$

is the class label.

The dataset is used to train the tree multiple times to analyze training time.

---



## 2. Procedure

1. For each scoring metric (`gini` and `entropy`), train the tree **multiple times** (e.g., 10 runs) on the same dataset.  
2. Record the **time taken** for each fit.  
3. Compute the **mean** and **standard deviation** of training times.

---




In [4]:
import time
import pandas as pd
import numpy as np
from scipy import stats


# Synthetic classification dataset
def generate_data(N=500, D=5, num_classes=2, random_seed=42):

    np.random.seed(random_seed)
    X = np.random.randn(N, D)
    y = np.random.randint(0, num_classes, size=N)
    return X, y

#  Experiment
def run(X, y, scoring='gini', n_runs=5, max_depth=10):

    times = []

    for _ in range(n_runs):
        tree = DecisionTree(max_depth=max_depth, scoring=scoring)
        start_time = time.time()
        tree.fit(X, y)
        end_time = time.time()
        times.append(end_time - start_time)

    return times

# Parameters
N = 1000        # Number of samples
D = 5           # Number of features
num_classes = 2 # Binary classification
n_runs = 10     

# Generate dataset
X, y = generate_data(N=N, D=D, num_classes=num_classes)

# Run experiments
gini_times  = run(X, y, scoring='gini', n_runs=n_runs)
entropy_times = run(X, y, scoring='entropy', n_runs=n_runs)


results = pd.DataFrame({
    'Scoring': ['Gini', 'Entropy'],
    'Mean_Time_sec': [np.mean(gini_times), np.mean(entropy_times)],
    'Std_Time_sec': [np.std(gini_times), np.std(entropy_times)],
})


In [5]:
results

,Scoring,Mean_Time_sec,Std_Time_sec
0,Gini,1.452875,0.003303
1,Entropy,1.648953,0.003220


---

## 3. Observations

- Training using **entropy** generally takes **longer** than **gini**.  
- This is expected because entropy involves computing **logarithms**, which are **computationally more expensive** than simple arithmetic used in Gini.  

---

## 4. Conclusion

- **Gini Impurity** is faster to compute and is often preferred for large datasets.  
- **Entropy** provides slightly more information theoretically but incurs extra computation due to the log operation.  
- In practice, **the difference in training time is noticeable** for large trees and high-dimensional datasets.  

> Entropy will almost always take more time than Gini due to the logarithmic computation.

---